In [1]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install gymnasium sb3-contrib stable-baselines3 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 2.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 5.8 MB/s eta 0:00:00


In [2]:
# ── 2. System path setup ─────────────────────────────────────────────────────
import sys, os
import numpy as np

DATASET_PATH = "/kaggle/input/datasets/nguyennhatphong/roguelike-rl-env"
sys.path.insert(0, DATASET_PATH)

print("Python version:", sys.version)
print("Dataset files:", os.listdir(DATASET_PATH)[:10])

Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Dataset files: ['python_roguelike']


In [ ]:
# ── 3. Balanced Environment Wrapper ──────────────────────────────────────────
from python_roguelike.env.roguelike_env import RoguelikeEnv
from python_roguelike.data.enums import GameState
from sb3_contrib.common.wrappers import ActionMasker


class BalancedEnv(RoguelikeEnv):
    """
    Balanced agent reward shaping (v4):
    - Floor reward: flat +25/floor (was floor × 1.2, peaked at ~18 — too weak)
    - Time penalty: −0.03/step (was −0.01 — too little urgency)
    - Enemy kill reward: +5 per kill (was missing entirely)
    - Removed ★1 card penalty (harmful early-game)
    - Win = +500, Loss = −200 (unified)
    """

    def __init__(self, seed=42, json_path=None, max_steps=10_000):
        super().__init__(seed=seed, json_path=json_path, max_steps=max_steps)
        self._prev_deck_size = 0
        self._prev_enemy_count = 0

    def reset(self, *, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self._prev_deck_size = len(self._controller.current_run.the_hero.deck.master_deck)
        self._prev_enemy_count = 0
        return obs, info

    def _count_alive_enemies(self):
        combat = self._controller.current_run.current_combat
        if combat is None:
            return 0
        return sum(1 for e in combat.enemies if e.current_health > 0)

    def step(self, action: int):
        run = self._controller.current_run
        prev_hp = run.the_hero.current_health
        prev_gold = run.the_hero.current_gold
        prev_floor = run.current_floor
        prev_alive = self._count_alive_enemies()

        obs, _base_reward, terminated, truncated, info = super().step(action)

        run = self._controller.current_run
        hero = run.the_hero
        reward = 0.0

        # HP — symmetric, moderate weight
        hp_delta = hero.current_health - prev_hp
        reward += hp_delta * 0.2

        # Gold — incentivize economy and smart shopping
        gold_delta = hero.current_gold - prev_gold
        if gold_delta > 0:
            reward += gold_delta * 0.02

        # Floor progress — flat +25 (was floor × 1.2, too weak)
        floor_delta = run.current_floor - prev_floor
        if floor_delta > 0:
            reward += floor_delta * 25.0

        # Enemy kill reward — direct combat signal
        curr_alive = self._count_alive_enemies()
        enemies_killed = prev_alive - curr_alive
        if enemies_killed > 0:
            reward += enemies_killed * 5.0

        # Deck quality reward — fires once when a new card is picked
        current_deck_size = len(hero.deck.master_deck)
        if current_deck_size > self._prev_deck_size:
            new_card = hero.deck.master_deck[-1]
            star = getattr(new_card, 'star_rating', 1)
            if star >= 4:
                reward += 8.0
            elif star == 3:
                reward += 3.0
            # ★1–2: no penalty (early-game forced picks)
            self._prev_deck_size = current_deck_size

        # Step cost — raised from −0.01 to −0.03 for meaningful urgency
        reward -= 0.03

        # Terminal rewards
        if terminated:
            if hero.current_health > 0:
                reward += 500.0
            else:
                reward -= 200.0

        self._prev_enemy_count = curr_alive
        return obs, reward, terminated, truncated, info


json_path = os.path.join(DATASET_PATH, "python_roguelike", "GameData.json")
env = BalancedEnv(seed=42, json_path=json_path)
obs, info = env.reset()
print("   BalancedEnv v4 created — obs shape:", obs.shape)
print("   Action space:", env.action_space)

2026-03-21 09:40:10.747646: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774086011.023468      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774086011.108163      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774086011.872308      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774086011.872360      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774086011.872363      55 computation_placer.cc:177] computation placer alr

   BalancedEnv created — obs shape: (153,)
   Action space: Discrete(67)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [4]:
# ── 4. Validate Action Masking ───────────────────────────────────────────────
masked_env = ActionMasker(env, lambda e: e.action_masks())
obs, info = masked_env.reset()

for step in range(200):
    mask = masked_env.action_masks()
    valid = np.where(mask)[0]
    obs, reward, terminated, truncated, info = masked_env.step(int(np.random.choice(valid)))
    if terminated or truncated:
        obs, info = masked_env.reset()

print("   Masking validation passed")

   Masking validation passed


In [5]:
# ── 5. Vectorized Environment ─────────────────────────────────────────────────
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor

N_ENVS = 4

def make_env(seed_offset: int):
    def _init():
        _env = BalancedEnv(seed=3000 + seed_offset, json_path=json_path, max_steps=10_000)
        return ActionMasker(_env, lambda e: e.action_masks())
    return _init

vec_env = VecMonitor(SubprocVecEnv([make_env(i) for i in range(N_ENVS)]))
print(f"   VecEnv ready: {N_ENVS} parallel envs")

2026-03-21 09:40:39.507562: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 09:40:39.507768: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 09:40:39.507956: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 09:40:39.509137: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774086039.534214     114 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already b

   VecEnv ready: 4 parallel envs


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.G

In [ ]:
# ── 6. MaskablePPO Model ─────────────────────────────────────────────────────
from sb3_contrib import MaskablePPO
from stable_baselines3.common.utils import get_linear_fn

policy_kwargs = dict(net_arch=dict(pi=[512, 512], vf=[512, 512]))

model = MaskablePPO(
    "MlpPolicy",
    vec_env,
    verbose=1,
    n_steps=2048,
    batch_size=512,
    n_epochs=10,
    learning_rate=get_linear_fn(3e-4, 1e-5, 1.0),
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.10,   # raised from 0.05 to prevent premature entropy collapse
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=policy_kwargs,
    tensorboard_log="/kaggle/working/tensorboard/balanced",
)
print("Model created. Policy params:", sum(p.numel() for p in model.policy.parameters()))

Using cpu device
Model created. Policy params: 717892


In [7]:
# ── 7. Callbacks ──────────────────────────────────────────────────────────────
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback, CallbackList

eval_env = ActionMasker(BalancedEnv(seed=7777, json_path=json_path), lambda e: e.action_masks())

callbacks = CallbackList([
    CheckpointCallback(
        save_freq=50_000,
        save_path="/kaggle/working/checkpoints/balanced",
        name_prefix="balanced_ppo",
        verbose=1,
    ),
    EvalCallback(
        eval_env,
        best_model_save_path="/kaggle/working/best_model/balanced",
        log_path="/kaggle/working/eval_logs/balanced",
        eval_freq=25_000,
        n_eval_episodes=15,
        deterministic=False,
        verbose=1,
    ),
])
print("   Callbacks configured")

   Callbacks configured


In [ ]:
# ── 8. Training ──────────────────────────────────────────────────────────────
TOTAL_TIMESTEPS = 5_000_000

print(f"   Starting Balanced Agent (v4) training for {TOTAL_TIMESTEPS:,} timesteps...")
print(f"   Floor: flat +25 (was scaling) | Step: −0.03 (was −0.01) | Kill: +5")
print(f"   Network: [512, 512], LR: linear_schedule(3e-4), ent_coef: 0.10")
print()

model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callbacks)

model.save("/kaggle/working/balanced_ppo_final")
print("\n   Training complete! Model saved to: /kaggle/working/balanced_ppo_final")

   Starting Balanced Agent training for 10,000,000 timesteps...
   Strategy: Equal offense/defense weighting
   Win = floor 15 cleared: +500 | Loss: −75 | Deck quality: +2×(star-1)
   Network: [512, 512], LR: linear_schedule(3e-4), ent_coef: 0.08

Logging to /kaggle/working/tensorboard/balanced/MaskablePPO_1


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_monitor.VecMonitor object at 0x7f6ed3e3eab0> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x7f6ed15e1f10>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 124      |
|    ep_rew_mean     | -39.9    |
| time/              |          |
|    fps             | 1301     |
|    iterations      | 1        |
|    time_elapsed    | 6        |
|    total_timesteps | 8192     |
---------------------------------


KeyboardInterrupt: 

In [ ]:
# ── 9. Evaluation ─────────────────────────────────────────────────────────────
model = MaskablePPO.load("/kaggle/working/best_model/balanced/best_model")

N_EVAL = 50
floors, full_wins, act_wins, hp_pcts, total_rewards = [], 0, 0, [], []

for ep in range(N_EVAL):
    ep_env = ActionMasker(BalancedEnv(seed=ep, json_path=json_path), lambda e: e.action_masks())
    obs, info = ep_env.reset()
    done = False
    ep_reward = 0.0

    while not done:
        action, _ = model.predict(obs, deterministic=True, action_masks=ep_env.action_masks())
        obs, reward, terminated, truncated, info = ep_env.step(int(action))
        ep_reward += reward
        done = terminated or truncated

    floors.append(info["floor"])
    total_rewards.append(ep_reward)
    if info["hp"] > 0 and terminated:
        hp_pcts.append(info["hp"] / info["max_hp"])
        if info["floor"] >= 15:
            full_wins += 1  # Full game clear
        else:
            act_wins += 1   # Survived but didn't finish

print(f"\n Balanced Agent Evaluation Results ({N_EVAL} episodes)")
print(f"═" * 50)
print(f"  Full-Game Win Rate:    {full_wins/N_EVAL*100:.1f}%  ({full_wins}/{N_EVAL}) — floor 15 cleared")
print(f"  Act Win Rate:          {act_wins/N_EVAL*100:.1f}%  ({act_wins}/{N_EVAL}) — survived but didn't finish")
print(f"  Death Rate:            {(N_EVAL-full_wins-act_wins)/N_EVAL*100:.1f}%")
print(f"  Avg Floor Reached:     {np.mean(floors):.1f} / 15")
print(f"  Max Floor Reached:     {max(floors)}")
print(f"  Avg Episode Reward:    {np.mean(total_rewards):.1f}")
if hp_pcts:
    print(f"  Avg HP% on Survive:    {np.mean(hp_pcts)*100:.1f}%")